# Exploratory Notebook — QSVM Fraud Detection
**Visualizing quantum kernels, feature maps, and ablation results**

> Paper: Ren & Zhang, IEEE CCPQT 2025 | ArXivist Stage 5

This notebook provides visual explorations of:
1. ZZFeatureMap quantum circuit structure (4/8/10-qubit variants)
2. Quantum kernel matrix heatmaps
3. Feature importance (KBest scores — paper Figure 3 reconstruction)
4. SMOTE synthesis visualization
5. Ablation results bar charts (Tables I & II reconstruction)
6. Confusion matrix reproduction (paper Figure 4)

> **Note:** Requires a fitted model at `checkpoints/qsvm_10qubit_qsmote.joblib`
> for sections 6–7. Sections 1–5 run without a pretrained checkpoint.

In [ ]:
import sys
print(f"Python: {sys.version}")

# Verify dependencies
for pkg in ["qiskit", "sklearn", "matplotlib", "seaborn", "numpy"]:
    try:
        m = __import__(pkg)
        print(f"  {pkg}: {getattr(m, '__version__', 'installed')} ✓")
    except ImportError:
        print(f"  {pkg}: NOT INSTALLED ✗ — run: pip install -r ../requirements.txt")

# All quantum simulation is CPU-based (cpu fallback always active)
print("\nDevice: CPU (all quantum simulation is CPU-based — no CUDA required ✓)")

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

print("Libraries loaded ✓")
from qsvm_fraud.utils.config import Config
Config.set_seed(42)

## 1. ZZFeatureMap Circuit Visualization

The ZZFeatureMap circuit $U_\phi(x)$ for varying qubit counts.
Each additional qubit exponentially expands the feature Hilbert space:
$\dim(\mathcal{H}) = 2^n$ (so 10 qubits → 1024-dimensional space).

In [ ]:
from qsvm_fraud.models.feature_map import QSVMFeatureMap

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
titles = ["4-qubit ZZFeatureMap (reps=2)", "8-qubit ZZFeatureMap (reps=2)", "10-qubit ZZFeatureMap (reps=2)"]

for ax, n_q, title in zip(axes, [4, 8, 10], titles):
    fm = QSVMFeatureMap(n_qubits=n_q, reps=2, entanglement="full")
    built = fm.build()
    ax.set_title(f"{title} | Hilbert space dim = {2**n_q}")
    ax.text(0.5, 0.5,
        f"ZZFeatureMap\nn_qubits={n_q}, reps=2, entanglement='full'\n"
        f"Parameters: {built.num_parameters} | "
        f"Hilbert space: 2^{n_q} = {2**n_q} dimensions",
        ha="center", va="center", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#e8f4f8", alpha=0.8),
        transform=ax.transAxes,
    )
    ax.axis("off")

plt.suptitle("ZZFeatureMap Configurations (Paper Section III-A, Eq. 3)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/feature_map_configs.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: /tmp/feature_map_configs.png")
print()
for n_q in [4, 8, 10]:
    fm = QSVMFeatureMap(n_qubits=n_q)
    built = fm.build()
    print(f"n_qubits={n_q:2d} | params={built.num_parameters:3d} | Hilbert dim={2**n_q:5d}")

## 2. Quantum Kernel Matrix Heatmap

Visualizing $K_{ij} = |\langle\phi(x_i)|\phi(x_j)\rangle|^2$ for a small synthetic dataset.
The kernel matrix encodes pairwise quantum similarity — higher values indicate more similar
feature representations in Hilbert space.

In [ ]:
import numpy as np
from qsvm_fraud.models.feature_map import QSVMFeatureMap
from qsvm_fraud.models.quantum_kernel import QSVMKernelComputer

np.random.seed(42)
n_samples = 12
# 6 "legit" samples + 6 "fraud" samples
X_legit = np.random.randn(6, 4) + np.array([1, 0, -1, 0.5])
X_fraud = np.random.randn(6, 4) + np.array([-1, 0.5, 1, -0.5])
X_demo = np.vstack([X_legit, X_fraud])
labels = ["L"]*6 + ["F"]*6  # L=legitimate, F=fraud

# Scale to reasonable range for quantum encoding
from sklearn.preprocessing import StandardScaler
X_demo = StandardScaler().fit_transform(X_demo)

fm = QSVMFeatureMap(n_qubits=4, reps=2, entanglement="full")
kc = QSVMKernelComputer(fm, cache_dir="/tmp/kernel_heatmap_demo/")

try:
    K = kc.compute_kernel_matrix(X_demo)
    print(f"Kernel matrix computed: {K.shape}")

    fig, ax = plt.subplots(figsize=(8, 6))
    im = sns.heatmap(
        K, annot=True, fmt=".2f", cmap="YlOrRd",
        xticklabels=labels, yticklabels=labels,
        ax=ax, vmin=0, vmax=1,
        cbar_kws={"label": r"$K(x_i, x_j) = |\langle\phi(x_i)|\phi(x_j)angle|^2$"},
    )
    ax.set_title(
        "Quantum Kernel Matrix — 4-qubit ZZFeatureMap\n"
        "(6 Legitimate + 6 Fraud synthetic samples)",
        fontweight="bold",
    )
    ax.set_xlabel("Sample")
    ax.set_ylabel("Sample")

    # Add class boundary line
    ax.axhline(y=6, color="blue", linewidth=2, linestyle="--", alpha=0.7)
    ax.axvline(x=6, color="blue", linewidth=2, linestyle="--", alpha=0.7)

    legend_handles = [
        mpatches.Patch(color="white", label="L = Legitimate (class 0)"),
        mpatches.Patch(color="white", label="F = Fraud (class 1)"),
        mpatches.Patch(color="white", linestyle="--", label="── Class boundary"),
    ]
    ax.legend(handles=legend_handles, loc="upper right", fontsize=9)

    plt.tight_layout()
    plt.savefig("/tmp/kernel_heatmap.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: /tmp/kernel_heatmap.png")

    print(f"\nMean intra-class similarity (L-L): {K[:6,:6].mean():.3f}")
    print(f"Mean intra-class similarity (F-F): {K[6:,6:].mean():.3f}")
    print(f"Mean inter-class similarity (L-F): {K[:6,6:].mean():.3f}")

except Exception as e:
    print(f"Kernel computation error: {e}")
    print("Install qiskit-aer to compute real quantum kernel values.")

## 3. Feature Importance Reconstruction (Paper Figure 3)

Reconstructing the KBest feature score ranking from paper Figure 3.
The paper selects the top-10 features by KBest score as input to the QSVM.

> **ASSUMPTION:** score_func=f_classif (confidence: 0.60) — shown here.
> The paper does not specify which KBest scoring function was used.

In [ ]:
import numpy as np
import pandas as pd

# Load real dataset if available; otherwise simulate feature scores
try:
    df = pd.read_csv("../data/raw/creditcard.csv")
    X = df.drop("Class", axis=1).values
    y = df["Class"].values
    feature_names = list(df.drop("Class", axis=1).columns)
    from sklearn.feature_selection import f_classif
    scores, pvalues = f_classif(X, y)
    source = "Computed from creditcard.csv"
except FileNotFoundError:
    # Simulate approximate scores based on paper Figure 3 visual
    # Top features visible in Figure 3: V28, V15, V13, V14, V17, V8, V12, V9, V19, V10
    np.random.seed(42)
    feature_names = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
    # Approximate scores from Figure 3 bar chart (visual estimation)
    base_scores = {
        "V28": 34000, "V15": 28000, "V13": 24000, "V14": 22000, "V17": 19000,
        "V8":  17000, "V12": 15500, "V9":  14000, "V19": 12000, "V10": 11000,
        "V5":   9000, "V11":  8000, "V4":   7500, "V3":   6000, "V6":   5000,
        "V7":   4000, "V2":   3500, "V1":   2500, "V16":  2000, "V22": 1500,
        "V20":  1200, "V21":  1000, "V23":   800, "V18":   700, "V27":  600,
        "V24":   400, "V26":   300, "V25":   200, "Time":  150, "Amount": 100,
    }
    scores = np.array([base_scores.get(f, np.random.randint(50, 500)) for f in feature_names])
    source = "Simulated (download creditcard.csv for real scores)"

# Sort features by score
df_scores = pd.DataFrame({"feature": feature_names, "score": scores})
df_scores = df_scores.sort_values("score", ascending=True)

# Identify top-10
top10 = df_scores.nlargest(10, "score")["feature"].tolist()

fig, ax = plt.subplots(figsize=(9, 10))
colors = ["#e74c3c" if f in top10 else "#3498db" for f in df_scores["feature"]]
bars = ax.barh(df_scores["feature"], df_scores["score"], color=colors)
ax.set_xlabel("KBest Score (f_classif)", fontsize=12)
ax.set_title(
    "Feature Scores Rank — Reconstruction of Paper Figure 3\n"
    "(red = selected top-10 features; blue = excluded)",
    fontweight="bold", fontsize=12,
)

red_patch = mpatches.Patch(color="#e74c3c", label=f"Top-10 selected: {', '.join(top10)}")
blue_patch = mpatches.Patch(color="#3498db", label="Excluded features")
ax.legend(handles=[red_patch, blue_patch], loc="lower right", fontsize=9)

# Mark the selection boundary
min_top10_score = df_scores[df_scores["feature"].isin(top10)]["score"].min()
ax.axvline(x=min_top10_score, color="red", linestyle="--", linewidth=1.5, alpha=0.7,
           label=f"Selection threshold: {min_top10_score:.0f}")

plt.tight_layout()
plt.savefig("/tmp/feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Source: {source}")
print(f"Top-10 selected features: {top10}")

## 4. Ablation Results — Tables I & II Visualization

Reproducing the paper's comparative results as bar charts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Table I: SMOTE vs. Undersampling (10-qubit) ─────────────────────────────
table1_data = {
    "Quantum-SMOTE": {"accuracy": 98.8, "f1": 0.962, "recall": 0.945, "auc": 0.992},
    "Undersampling": {"accuracy": 92.4, "f1": 0.907, "recall": 0.881, "auc": 0.972},
}

# ── Table II: QSVM vs. SVM across qubit counts ──────────────────────────────
table2_data = {
    "QSVM-4q":  {"accuracy": 92.2, "f1": 0.902, "recall": 0.894, "auc": 0.971, "type": "QSVM"},
    "QSVM-8q":  {"accuracy": 96.7, "f1": 0.956, "recall": 0.941, "auc": 0.983, "type": "QSVM"},
    "QSVM-10q": {"accuracy": 98.8, "f1": 0.962, "recall": 0.945, "auc": 0.992, "type": "QSVM"},
    "SVM-4f":   {"accuracy": 91.4, "f1": 0.897, "recall": 0.871, "auc": 0.969, "type": "SVM"},
    "SVM-8f":   {"accuracy": 94.3, "f1": 0.932, "recall": 0.921, "auc": 0.973, "type": "SVM"},
    "SVM-10f":  {"accuracy": 95.1, "f1": 0.933, "recall": 0.923, "auc": 0.979, "type": "SVM"},
}

metrics_display = ["accuracy", "f1", "recall", "auc"]
labels_display = ["Accuracy (%)", "F1-score", "Recall", "AUC"]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("Paper Results — Tables I & II Reconstruction\n(Ren & Zhang, IEEE CCPQT 2025)",
             fontsize=14, fontweight="bold")

# Table I
for col, (metric, mlabel) in enumerate(zip(metrics_display, labels_display)):
    ax = axes[0, col]
    names = list(table1_data.keys())
    vals = [table1_data[n][metric] for n in names]
    colors_t1 = ["#e74c3c", "#3498db"]
    bars = ax.bar(names, vals, color=colors_t1, edgecolor="black", linewidth=0.5)
    ax.bar_label(bars, fmt=lambda v: f"{v:.3f}" if metric != "accuracy" else f"{v:.1f}%",
                 padding=2, fontsize=9)
    ax.set_title(f"Table I: {mlabel}", fontweight="bold", fontsize=10)
    ax.set_ylim(min(vals)*0.95, max(vals)*1.06)
    ax.tick_params(axis="x", rotation=15)

# Table II
qsvm_color = "#e74c3c"
svm_color = "#3498db"
for col, (metric, mlabel) in enumerate(zip(metrics_display, labels_display)):
    ax = axes[1, col]
    names = list(table2_data.keys())
    vals = [table2_data[n][metric] for n in names]
    colors_t2 = [qsvm_color if table2_data[n]["type"] == "QSVM" else svm_color for n in names]
    bars = ax.bar(names, vals, color=colors_t2, edgecolor="black", linewidth=0.5)
    ax.bar_label(bars, fmt=lambda v: f"{v:.3f}" if metric != "accuracy" else f"{v:.1f}%",
                 padding=2, fontsize=8)
    ax.set_title(f"Table II: {mlabel}", fontweight="bold", fontsize=10)
    ax.set_ylim(min(vals)*0.95, max(vals)*1.06)
    ax.tick_params(axis="x", rotation=20, labelsize=8)

# Legend
qsvm_patch = mpatches.Patch(color=qsvm_color, label="QSVM (quantum)")
svm_patch  = mpatches.Patch(color=svm_color,  label="SVM (classical)")
smote_patch = mpatches.Patch(color=qsvm_color, label="Quantum-SMOTE")
under_patch  = mpatches.Patch(color=svm_color,  label="Undersampling")
axes[0, 3].legend(handles=[smote_patch, under_patch], loc="lower right", fontsize=9)
axes[1, 3].legend(handles=[qsvm_patch, svm_patch],   loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/ablation_charts.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: /tmp/ablation_charts.png")

## 5. QSVM vs Classical SVM Performance Gap

The paper claims the QSVM advantage **grows with qubit count** — demonstrating quantum scalability.
This visualization quantifies that gap across all three configurations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

qubit_counts = [4, 8, 10]
qsvm_acc = [92.2, 96.7, 98.8]
svm_acc  = [91.4, 94.3, 95.1]
gaps     = [q - s for q, s in zip(qsvm_acc, svm_acc)]

qsvm_auc = [0.971, 0.983, 0.992]
svm_auc  = [0.969, 0.973, 0.979]
auc_gaps = [q - s for q, s in zip(qsvm_auc, svm_auc)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy comparison
ax = axes[0]
x = np.array(qubit_counts)
ax.plot(x, qsvm_acc, "o-", color="#e74c3c", linewidth=2.5, markersize=9, label="QSVM", zorder=3)
ax.plot(x, svm_acc,  "s--", color="#3498db", linewidth=2.5, markersize=9, label="Classical SVM", zorder=3)
for xi, gq, gs, gap in zip(x, qsvm_acc, svm_acc, gaps):
    ax.annotate(f"+{gap:.1f}pp", xy=(xi, (gq+gs)/2), xytext=(xi+0.15, (gq+gs)/2),
                color="darkgreen", fontweight="bold", fontsize=10)
    ax.fill_between([xi, xi], [gs, gq], alpha=0.2, color="green")
ax.set_xlabel("Number of Qubits / Features", fontsize=12)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("QSVM vs SVM Accuracy\n(Table II — widening gap)", fontweight="bold")
ax.set_xticks(qubit_counts)
ax.set_xticklabels(["4
(4-feat)", "8
(8-feat)", "10
(10-feat)"])
ax.legend(fontsize=11)
ax.set_ylim(89, 100.5)
ax.grid(True, alpha=0.3)

# AUC comparison
ax = axes[1]
ax.plot(x, qsvm_auc, "o-", color="#e74c3c", linewidth=2.5, markersize=9, label="QSVM")
ax.plot(x, svm_auc,  "s--", color="#3498db", linewidth=2.5, markersize=9, label="Classical SVM")
ax.axhline(y=1.0, color="gray", linestyle=":", linewidth=1, label="Perfect classifier")
for xi, gq, gs, gap in zip(x, qsvm_auc, svm_auc, auc_gaps):
    ax.annotate(f"+{gap:.3f}", xy=(xi, (gq+gs)/2+0.001), xytext=(xi+0.15, (gq+gs)/2+0.001),
                color="darkgreen", fontweight="bold", fontsize=10)
ax.set_xlabel("Number of Qubits / Features", fontsize=12)
ax.set_ylabel("AUC", fontsize=12)
ax.set_title("QSVM vs SVM AUC\n(approaching perfect classifier = 1.0)", fontweight="bold")
ax.set_xticks(qubit_counts)
ax.set_xticklabels(["4
(4-feat)", "8
(8-feat)", "10
(10-feat)"])
ax.legend(fontsize=11)
ax.set_ylim(0.960, 1.005)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/qsvm_vs_svm_gap.png", dpi=120, bbox_inches="tight")
plt.show()

print("Key finding from paper (Section IV-B):")
print(f"  Gap at 4-qubit:  {gaps[0]:.1f} percentage points")
print(f"  Gap at 8-qubit:  {gaps[1]:.1f} percentage points")
print(f"  Gap at 10-qubit: {gaps[2]:.1f} percentage points")
print(f"\n→ The performance advantage grows with quantum resources — demonstrating scalability.")